Colin Klick

October 30, 2026

The following program prints the predicted phenotype given a sample ID or barcode of a sequenced cat.

On the colored portion of a cat, the predicted phenotype will specify dilute, colorpoint, and nonagouti if the variants predict this.  If the cat is predicted to be agouti, the pattern will also be specified as mackerel, blotched (classic), or ticked.

In [1]:
import os
os.environ["PATH"] = "/modules/spack/packages/linux-ubuntu24.04-x86_64_v3/gcc-13.2.0/bcftools-1.19-h2elsokvbdpuswb25fywovhs5gbm572w/bin:" + os.environ["PATH"]
import io, os, shutil, subprocess
import pandas as pd

if shutil.which("bcftools") is None:
    raise RuntimeError("bcftools not on PATH. Run `module load bcftools` BEFORE launching Jupyter.")

BCF_FILE = "/scratch4/workspace/bryc_broadinstitute_org-darwins_cats/merged_data/2026-06-23_merged_darwins_cats_only.vcf.gz"
REGIONS  = os.path.expanduser("~/all_sites.txt")

SITE_NAMES = {
    ('chrD1', 44013031): 'siamese',
    ('chrD1', 44022819): 'burmese',
    ('chrA3', 24831082): 'agouti',
    ('chrA1', 94477275): 's59x',
    ('chrA1', 94536796): 'w841x',
    ('chrC1', 218197447): 'dilute',
    ('chrB1', 40428846): 'a18v',
    ('chrB1', 40429492): 'c63y',
    ('chrD4', 38142265): 'chocolate',
    ('chrD4', 38130163): 'cinnamon',
    ('chrA1', 94477515): 't139n',
    ('chrX', 108588004): 'orange',
}

In [2]:
def count_alt_alleles(gt):
    if gt is None or pd.isna(gt):
        return pd.NA
    alleles = gt.replace('|', '/').split('/')
    if '.' in alleles:
        return pd.NA
    return sum(int(a) > 0 for a in alleles)

def get_cat_genotypes(sample_id):
    cmd = ["bcftools", "query", "-s", sample_id, "-R", REGIONS,
           "-f", "%CHROM\t%POS\t[%GT]\n", BCF_FILE]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"bcftools failed (is '{sample_id}' in the VCF?):\n{result.stderr}")
    df = pd.read_csv(io.StringIO(result.stdout), sep="\t",
                     names=["chrom", "pos", "gt"], dtype={"gt": str})
    df["site"] = [SITE_NAMES.get((c, p), f"{c}_{p}") for c, p in zip(df.chrom, df.pos)]
    df["allele_count"] = df["gt"].map(count_alt_alleles)
    found = df.set_index("site")["allele_count"].to_dict()
    # sites absent from the VCF show up as NA instead of disappearing
    return {name: found.get(name, pd.NA) for name in SITE_NAMES.values()}

In [3]:
def is_n(x, n):
    """True only if x is a known value equal to n (NA -> False)."""
    return (not pd.isna(x)) and x == n

def classify_ticked(g):
    a, c = g['a18v'], g['c63y']
    if is_n(a, 2) or is_n(c, 2) or (is_n(a, 1) and is_n(c, 1)):
        return "ticked"
    if pd.isna(a) or pd.isna(c):
        return "unknown"
    if a + c == 1:
        return "single_copy"
    return "not_ticked"          # both 0

def classify_blotched(g):
    # Taqpep loss-of-function alleles: recessive, so need 2 hits total
    # (homozygous at one site, or compound heterozygous across sites)
    known = [g[k] for k in ('s59x', 'w841x') if not pd.isna(g[k])]
    if sum(known) >= 2:
        return "blotched"
    if len(known) < 2:
        return "unknown"
    return "not_blotched"

In [4]:
def call_phenotype(g):
    parts = []
    if g['dilute'] == 2:
        parts.append("dilute")
    if g['siamese'] == 2 or g['burmese'] == 2:
        parts.append('colorpoint')
    if g['agouti'] == 2:
        parts.append('nonagouti')
    else:
        parts.append('agouti')
        if not pd.isna(g['agouti']):       # agouti is 0 or 1
            ticked = classify_ticked(g)
            if ticked == "ticked":
                parts.append("ticked")
            elif ticked == "unknown":
                parts.append("pattern unknown (ticked loci missing)")
            else:
                blotched = classify_blotched(g)
                if blotched == "blotched":
                    pattern = "blotched"
                elif blotched == "unknown":
                    pattern = "mackerel or blotched (Taqpep loci missing)"
                else:
                    pattern = "mackerel"
                parts.append(pattern)
    
    text = ", ".join(parts)
    return text

In [6]:
sample_id = "DA00000645"
my_dict = get_cat_genotypes(sample_id)
print(my_dict)
print(call_phenotype(my_dict))

{'siamese': 0, 'burmese': 0, 'agouti': 1, 's59x': 0, 'w841x': 2, 'dilute': 2, 'a18v': 0, 'c63y': 0, 'chocolate': 2, 'cinnamon': 0, 't139n': 2, 'orange': 2}
dilute, agouti, blotched
